In [1]:
import numpy as np
import pandas as pd


def fitted_loglikes(y, x):
    """Exactly the two fitted unit-covariance Gaussian log-likelihoods."""
    theta = y.mean()
    beta = x.mean()
    const = -np.log(2.0 * np.pi)

    ll1 = const - 0.5 * ((y - theta)**2 + x**2)
    ll2 = const - 0.5 * (y**2 + (x - beta)**2)
    return ll1, ll2


def sw_parts(a, b, epsilon):
    """
    Your uncorrected numerator and denominator.
    Works with shape (n,) or (B, n).
    """
    n = a.shape[-1]
    if n % 2:
        raise ValueError("Use an even sample size.")

    d = a - b
    D = d.sum(axis=-1)
    S = a[..., ::2].sum(axis=-1) - b[..., 1::2].sum(axis=-1)
    numerator = D + epsilon * S

    variance = (
        (1.0 + epsilon) * d.var(axis=-1, ddof=0)
        + (epsilon**2 / 2.0) * (
            a[..., ::2].var(axis=-1, ddof=0)
            + b[..., 1::2].var(axis=-1, ddof=0)
        )
    )
    denominator = np.sqrt(n * variance)

    if np.any(~np.isfinite(denominator)) or np.any(denominator <= 0):
        raise FloatingPointError("Invalid studentizer; inspect this replication.")

    return numerator, denominator


def one_replication(rng, n=200, B=1000, epsilon=0.5,
                    alpha=0.05, mu_y=0.0, mu_x=0.0):
    y = rng.normal(mu_y, 5.0, n)
    x = rng.normal(mu_x, 1.0, n)
    a, b = fitted_loglikes(y, x)

    N, denominator = sw_parts(a, b, epsilon)
    observed = N / denominator
    D = (a - b).sum()
    singleton_center = (1.0 + epsilon / 2.0) * D

    # Identical singleton draws for the old and corrected methods.
    indices = rng.integers(0, n, size=(B, n))
    Ns, ds = sw_parts(a[indices], b[indices], epsilon)

    old_singleton = (Ns - N) / ds
    corrected_singleton = (Ns - singleton_center) / ds

    # Pairwise bootstrap, preserving parity.
    pairs = rng.integers(0, n // 2, size=(B, n // 2))
    pair_indices = np.empty((B, n), dtype=int)
    pair_indices[:, ::2] = 2 * pairs
    pair_indices[:, 1::2] = 2 * pairs + 1

    Np, dp = sw_parts(a[pair_indices], b[pair_indices], epsilon)
    pairwise = (Np - N) / dp

    roots = (old_singleton, corrected_singleton, pairwise)
    rejected = []
    for root in roots:
        lower, upper = np.quantile(root, [alpha / 2, 1 - alpha / 2])
        rejected.append(observed < lower or observed > upper)

    return rejected


def centering_experiment(R=1000, B=1000, n=200, epsilon=0.5,
                         alpha=0.05, mu_y=0.0, mu_x=0.0,
                         seed=20261015):
    rng = np.random.default_rng(seed)
    rejected = np.empty((R, 3), dtype=bool)

    for r in range(R):
        rejected[r] = one_replication(
            rng, n=n, B=B, epsilon=epsilon, alpha=alpha,
            mu_y=mu_y, mu_x=mu_x
        )

    rates = rejected.mean(axis=0)
    result = pd.DataFrame({
        "method": [
            "Singleton: original centering",
            "Singleton: corrected centering",
            "Pairwise: original centering",
        ],
        "rejection_rate": rates,
        "MC_standard_error": np.sqrt(rates * (1.0 - rates) / R),
    })

    # Paired Monte Carlo comparison: same original datasets.
    difference = rejected[:, 1].astype(float) - rejected[:, 2].astype(float)
    comparison = {
        "corrected_singleton_minus_pairwise": difference.mean(),
        "paired_MC_standard_error": difference.std(ddof=1) / np.sqrt(R),
    }
    return result, comparison


# R1's diagnostic setting: exact equivalence null, n=200, epsilon=.5.
results, comparison = centering_experiment()
print(results.to_string(index=False))
print(comparison)

                        method  rejection_rate  MC_standard_error
 Singleton: original centering           0.313           0.014664
Singleton: corrected centering           0.053           0.007085
  Pairwise: original centering           0.062           0.007626
{'corrected_singleton_minus_pairwise': -0.009, 'paired_MC_standard_error': 0.0045760179129828524}
